EXPLORACIÓN DE DATOS, GRÁFICAS Y DEMÁS -> MACARENA Y MIGUEL

# 02 — Enrichment

## [ENG] Objectives

Walk through the enrichment process step by step:

1. **Merge sensors** — Combine VIIRS + MODIS into a single dataset
2. **Confidence mapping** — Reconcile numerical (VIIRS) and categorical (MODIS) confidence
3. **CLC enrichment** — Add land cover class to each fire detection
4. **Weather enrichment** — Add historical weather data to each fire detection

Each step calls functions from `src/wildfire/` and shows the resulting DataFrame.

## [ESP] Objetivos

Recorrer el proceso de enriquecimiento paso a paso:

1. **Fusión de sensores** — Combinar VIIRS + MODIS en un solo dataset
2. **Mapeo de confianza** — Reconciliar confianza numérica (VIIRS) y categórica (MODIS)
3. **Enriquecimiento CLC** — Añadir clase de cobertura del suelo a cada detección
4. **Enriquecimiento climático** — Añadir datos de clima histórico a cada detección

Cada paso llama funciones de `src/wildfire/` y muestra el DataFrame resultante.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r'C:\Users\Macarena\OneDrive\Documentos\wildfires-project\data\processed\enriched\firms_spain_final.csv')

# DESCOMPOSICIÓN TEMPORAL Y TRIGONOMETRÍA
df['acq_date'] = pd.to_datetime(df['acq_date'])

df['mes'] = df['acq_date'].dt.month
df['dia_del_año'] = df['acq_date'].dt.dayofyear
df['es_fin_semana'] = df['acq_date'].dt.dayofweek.isin([5, 6]).astype(int)

# Aplicación de senos y cosenos
df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)
df['dia_año_sin'] = np.sin(2 * np.pi * df['dia_del_año'] / 365.25) 
df['dia_año_cos'] = np.cos(2 * np.pi * df['dia_del_año'] / 365.25)

# Dividir en franjas horarias
df['acq_time_str'] = df['acq_time'].astype(str).str.zfill(4)
df['hora'] = pd.to_datetime(df['acq_time_str'], format='%H%M', errors='coerce').dt.hour

bins_hora = [-1, 6, 12, 18, 24]
labels_hora = ['Madrugada', 'Mañana', 'Tarde_Pico_Calor', 'Noche']
df['franja_horaria'] = pd.cut(df['hora'], bins=bins_hora, labels=labels_hora, right=False)

# Superficie forestal aproximada en hectáreas (MITECO)
superficie_forestal_ha = {
    'Andalucía': 4400000, 'Aragón': 2600000, 'Asturias': 770000,
    'Islas Baleares': 220000, 'Canarias': 570000, 'Cantabria': 360000,
    'Castilla y León': 5100000, 'Castilla-La Mancha': 3600000, 'Cataluña': 2000000,
    'Comunidad Valenciana': 1300000, 'Extremadura': 2700000, 'Galicia': 2000000,
    'Madrid': 430000, 'Región de Murcia': 510000, 'Navarra': 610000,
    'País Vasco': 490000, 'La Rioja': 340000
}

# Mapear las hectáreas a cada fila
df['sup_forestal_ha'] = df['ccaa'].map(superficie_forestal_ha)

# Ratios
df['ratio_prevencion_extincion'] = df['sum_prevention'] / (df['sum_extinction'] + 1e-5)
df['euros_prevencion_por_ha'] = (df['sum_prevention'] * 1000000) / df['sup_forestal_ha']
df['euros_extincion_por_ha'] = (df['sum_extinction'] * 1000000) / df['sup_forestal_ha']

# Limpieza de variables usadas que ya no necesitamos
df = df.drop(columns=['acq_time_str', 'hora', 'sup_forestal_ha'])

# 1. visualizar la transformación cíclica
display(df[['acq_date', 'mes', 'mes_sin', 'mes_cos']].head(5))

# 2. Ver la relativización económica (estadísticos básicos)
display(df[['sum_prevention', 'euros_prevencion_por_ha', 'ratio_prevencion_extincion']].describe())

# 3. Comprobar si las nuevas variables correlacionan mejor con la severidad del incendio (FRP) (esto no hace falta en verdad pero bueno poruqe se viera algo)
correlaciones = df.corr(method='spearman', numeric_only=True)['frp'].sort_values(ascending=False)
display(correlaciones)

,acq_date,mes,mes_sin,mes_cos
0,2023-01-01,1,0.5,0.866025
1,2023-01-01,1,0.5,0.866025
2,2023-01-01,1,0.5,0.866025
3,2023-01-02,1,0.5,0.866025
4,2023-01-03,1,0.5,0.866025


,sum_prevention,euros_prevencion_por_ha,ratio_prevencion_extincion
count,35282.000000,35282.000000,35282.000000
mean,112.424372,58.583335,0.651304
std,91.200840,29.588702,0.170657
min,6.500000,15.269231,0.404922
25%,66.000000,32.962963,0.511628
50%,80.000000,50.000000,0.611888
75%,148.000000,85.714286,0.695455
max,319.000000,166.326531,0.990683


frp                           1.000000
confidence_og_num             0.730410
brightness                    0.656574
acq_time                      0.494901
et0_fao_evapotranspiration    0.466065
brightness_ir                 0.453114
shortwave_radiation           0.447105
is_day                        0.432000
version                       0.401383
boundary_layer_height         0.398027
vapour_pressure_deficit       0.385531
scan                          0.366442
wind_gusts_10m                0.337647
track                         0.316514
confidence_num                0.314614
temperature_2m                0.246688
wind_speed_10m                0.162718
clc_uniform_surroundings      0.102893
rain                          0.039644
precipitation                 0.039485
dia_del_año                   0.003884
mes                           0.001485
dia_año_sin                  -0.003428
es_fin_semana                -0.007074
euros_extincion_por_ha       -0.023069
mes_sin                  